In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% ! important;}
div.cell.code_cell.rendered{width:100%}
div.input_prompt{padding:0px}
div.CodeMirror {font-family:Consolas ; font-size:12pt;}
div.text_cell_render.rendered_html {font-size:12pt;}
div.output {font-size:12pt; font-weight:bold}
div.input {font-family:Consolas ; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# 벡터 DB : Chroma VS Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB
- Pinecone : 클라우드 vector DB
    (https://www.pinecone.io/ 에서 api key 생성 -> .env에 추가(PINECONE_API_KEY 등록)

# 0. 패키지 설치

In [3]:
%pip install -q pinecone langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


# 1. knowledge Base 구성을 위한 데이터 생성

In [2]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('data/소득세법(법률)(제21065호)(20260102).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)
document_list=loader.load_and_split(text_splitter=text_splitter)
len(document_list)

193

In [3]:
# embedding : upstage의 solar-embedding-1-large-passage
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(model="solar-embedding-1-large-passage")

In [5]:
len(embedding.embed_query('소득세법'))

4096

In [7]:
%%time
#pinecone vector database 저장
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
pc= Pinecone(
    api_key=os.getenv("PINECONE_API_KEY")
)
# 데이터를 처음 업로드할 때 
# index_name="tax-index-upstage"
# database = PineconeVectorStore.from_documents(
#     documents = document_list,
#     embedding= embedding,
#     index_name=index_name
# )

# 업로드시 경고가 안보이려면 아나콘다 프롬프트 llm 환경에서 conda install -c conda-forge ipywidgets

# 업로드한 벡터db를 가져올 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)

CPU times: total: 0 ns
Wall time: 0 ns


# 2. 답변 생성을 위한 Retrival

In [8]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrueved_docs = database.similarity_search(query, k=3)

In [9]:
# retrueved_docs[0].page_content
retrueved_doc = "\n\n--\n\n".join([doc.page_content for doc in retrueved_docs])

In [10]:
# query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
# retrueved_docs = database.similarity_search(query, k=3) 와 아래코드는 동일함

retruever = database.as_retriever(
    search_kwargs={"k":3}
)
retrueved_docs = retruever.invoke(query)

# 3. 답변 생성

In [ ]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4.1-namo")

In [11]:
# upstate에서 받은 달러로 llm을 사용하고 싶다면
from langchain_upstage import ChatUpstage
llm = ChatUpstage(
    model="solar-pro2",
    reasoning_effort="high" # 느리지만 더 깊게 추론함(low, medium)
)

In [12]:
prompt = f"""
- 당신은 최고의 한국 소득세법 전문가입니다.
- [context]를 참고해서 사용자의 질문에 답변해 주세요.
- [context]는 다음과 같아요
{retrueved_doc}
- 질문 : {query}"""

In [13]:
ai_message = llm.invoke(prompt)

In [22]:
# 08 답변
print(ai_message.content)

연봉 5,000만원인 직장인의 소득세 계산은 다음과 같이 진행됩니다. 단, 추가적인 공제 항목(의료비, 보험료, 기부금 등)은 고려하지 않았습니다.

---

### **1. 근로소득공제 계산**
- **총급여액**: 5,000만원  
- **근로소득공제**:  
  - 4,500만원 초과 ~ 1억원 이하 구간 적용  
  - 공제액 = 1,125만원 + (5,000만원 - 4,500만원) × 5% = **1,150만원**  
- **과세표준**: 5,000만원 - 1,150만원 = **3,885만원**

---

### **2. 종합소득세 계산 (누진세율 적용)**
- **세율 구간**:  
  - 1,200만원 이하: 6%  
  - 1,200만원 초과 ~ 4,600만원 이하: 15% (초과분에 적용)  
- **계산**:  
  - 1,200만원 × 6% = **72만원**  
  - (3,885만원 - 1,200만원) × 15% = 2,685만원 × 15% = **402.75만원**  
  - **산출세액**: 72만원 + 402.75만원 = **474.75만원**

---

### **3. 근로소득세액공제 적용**
- **공제율**:  
  - 산출세액이 132만원 초과 시: 102만원 + (산출세액 - 132만원) × 20% (최대 110만원)  
- **계산**:  
  - 102만원 + (474.75만원 - 132만원) × 20% = 102만원 + 685.5만원 × 20% = **102만원 + 137.1만원 = 239.1만원**  
  - **최대 한도 110만원 적용**  
- **공제 후 세액**: 474.75만원 - 110만원 = **364.75만원**

---

### **4. 지방소득세 (주민세)**
- **계산**: 소득세액의 10%  
- **금액**: 364.75만원 × 10% = **36.475만원**

---

### **최종 납부세액**
- **국세(소득세)**: **364.75만원**  
- **지방세(주민세)**: **36

In [14]:
# 09 답변
print(ai_message.content)

한국 소득세법 및 제공된 [context]를 기반으로 연봉 5천만원인 직장인의 소득세를 계산하는 과정은 다음과 같습니다. 단, **자녀 수, 연금계좌 납입액, 추가 공제 항목 등 구체적 정보가 없으므로 기본 가정 하에 계산**하였습니다.

---

### 1. **근로소득공제 (제47조)**
- 총급여액 5,000만원에서 근로소득공제 적용  
  (※ 정확한 공제액은 [context]에 명시되지 않아 **표준 공제율 추정** 활용)  
  - 예시 공제 계산 (실제 법령 별표 기준):  
    - 500만원 이하: 70% (350만원)  
    - 500만원~1,500만원: 40% (400만원)  
    - 1,500만원~3,000만원: 15% (225만원)  
    - 3,000만원~5,000만원: 10% (200만원)  
    - **총 공제액: 350 + 400 + 225 + 200 = 1,175만원**  
    - 단, 최대 공제액 한도 2,000만원 미만이므로 **1,175만원 공제**  
  - **과세표준: 5,000만원 - 1,175만원 = 3,825만원**

---

### 2. **종합소득산출세액 계산**
- **누진세율 적용** (2024년 기준):  
  - 1,200만원 이하: 6% → 72만원  
  - 1,200만원~4,600만원: 15% → (3,825만원 - 1,200만원) × 15% = **393.75만원**  
  - **산출세액: 72만원 + 393.75만원 = 465.75만원**

---

### 3. **세액공제 적용**
#### (1) **근로소득세액공제 (제59조)**
- 총급여액 5,000만원 (3,300만원 초과 ~ 7,000만원 이하 구간):  
  - 공제액 = 74만원 - [(5,000만원 - 3,300만원) × 0.8%]  
    = 74만원 - (1,700만원 × 0.008) = **74만원 - 1.36만원 = 72.64만원**  
  - 최소 66만원 적용 조건 충족 → **72.64만원 공

# 4. langchain 전달

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_upstage import ChatUpstage
from dotenv import load_dotenv
load_dotenv()

promptTemplate = ChatPromptTemplate([
    ("system","당신은 최고의 한국 소득세 전문가 입니다."),
    ("human", f"""다음 문맥을 참고하여 질문에 답변하세요.
    답을 모르면 모른다고 답변하세요.
    최대 3문장으로 간결하게 답변하세요.
    질문 :{{question}}
    문맥 :{{context}}
    """)
])

In [18]:
prompt = promptTemplate.invoke({
    'context':retrueved_docs,
    'question':query
})

In [ ]:
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()
output_parser.invoke(llm.invoke(promptTemplate.invoke({
    'context':retrieval_doc,
    'question' : query
})))

In [ ]:
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

# 1. LLM과 임베딩 초기화
load_dotenv()
llm = ChatUpstage(model="solar-pro2")
embedding = UpstageEmbeddings(model="solar-embedding-1-large-passage")
# 2. vector store load
vectorstore = (
    embedding_function=embedding,
    collection_name="tax-collection",
    persist_directory="./chroma_upstage/"
)
# 3. Retriever 생성
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":4}
)
# 4. 프롬프트 템플릿
template = f"""당신은 최고의 한국 소득세 전문가입니다.
다음 문맥을 참고하여 질문에 답하세요
답을 모르면 모른다고 답하세요
최대 3문장으로 간결하게 답변하세요.
질문:{{query}}
문맥:{{context}}
답변:"""
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 document를 텍스트로 변환하는 함수
def format_documents(documents):
    return "\n\n--\n\n".join([doc.page_content for doc in documents])

In [ ]:
# 6. RAG 체인 구성(LCEL 방식)
from langchain_core.runnables import RunnablePassthrough # {"query":"~"}=>"~"
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문 그대로 전달
    }
    | prompt # prompt에 cdontext와 query 변수 주입
    | llm 
    | StrOutputParser()
)
# 7. 실행
query ="연봉 5천만원인 직장인의 소득세는 얼마인가요?"
rag_chain.invoke(query)